# R2-CAP · Capstone — modelo predictivo de punta a punta — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 17–18 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Plantear un **problema de ML** al servicio de una decisión pública.
2. Elegir **features sin fuga** y una **partición temporal** honesta.
3. Entrenar un pipeline y **evaluarlo contra una línea base**.
4. Documentarlo en una **ficha de modelo** y un **brief** para quien decide.

**Competencia de salida:** ejecutar el ciclo de vida de un modelo de forma responsable, de los datos a la ficha y el brief.

**Dato real:** PM2.5 diario de 14 estaciones SINCA (ejemplo) o el dataset de tu organismo.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "aire_diario.csv"          # ← reemplázalo por el dataset de tu organismo
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)

df = pd.read_csv(CSV, parse_dates=["fecha"])
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

## 1. Enmarca el problema de modelado

Antes de tocar un modelo, escribe en una frase la **decisión pública** que el modelo va a apoyar
y por qué un modelo aporta algo que una regla simple no. Ejemplo guía:

> *Anticipar, con un día de aviso, los días en que una estación de monitoreo superará la norma
> diaria de PM2.5 (50 µg/m³, D.S. 12/2011 del MMA), para que el equipo de gestión ambiental
> prepare medidas preventivas y avisos a la población — **con revisión humana**.*

Dos preguntas que el capstone debe responder con honestidad: **¿le gana a la regla simple?**
(«si hoy superó la norma, mañana también») y **¿para quién funciona peor?**

**Elige tu dataset** (en este orden de preferencia):

1. **Datos de tu propio organismo** — el capstone vale el doble si responde una pregunta real
   de tu servicio (con los resguardos de datos personales que corresponden).
2. El **banco del bootcamp** (todos reales y ya curados, en `data/` del repositorio):
   `aire_diario.csv` + `aire_nuevos.csv` (PM2.5 diario por estación SINCA, 2020–2025 y 2026) ·
   `sismos.csv` (USGS M≥4,5, 2020–2025) · `matricula.csv` / `establecimientos.csv` (MINEDUC 2025) ·
   `reclamos.csv` (SERNAC, histórico 2011) · `fichas.csv` (trámites ChileAtiende) ·
   `comunas.csv` · `parques.csv`.
3. Cualquier dataset de **datos.gob.cl** (ya sabes usar su API, R1-01).

> Esta plantilla usa `aire_diario.csv` como ejemplo; reemplázalo por el tuyo y adapta cada paso.

### Tabla de modelado (ejemplo ya hecho)

Una fila = **una estación en un día**. Las *features* describen lo que se sabe **hoy**; el
objetivo es lo que pasará **mañana**. Fíjate que la tabla trae, a propósito, dos columnas del
futuro (`pm25_manana` y `episodio_manana`): son la respuesta, y una de ellas es una trampa.

In [ ]:
def construir_tabla(df):
    partes = []
    for estacion, d in df.groupby("estacion"):
        d = d.set_index("fecha").sort_index().asfreq("D")      # calendario completo
        d["estacion"] = estacion
        d["pm25_ayer"] = d["pm25"].shift(1)
        d["media_7d"] = d["pm25"].rolling(7, min_periods=5).mean()
        d["mes"] = d.index.month
        d["pm25_manana"] = d["pm25"].shift(-1)                  # ⚠️ el futuro
        partes.append(d.reset_index())
    t = pd.concat(partes, ignore_index=True)
    t = t.dropna(subset=["pm25", "pm25_ayer", "media_7d", "pm25_manana"]).reset_index(drop=True)
    t["episodio_manana"] = (t["pm25_manana"] > 50).astype(int)  # objetivo: 1 si mañana supera la norma
    return t

tabla = construir_tabla(df)
print("filas:", len(tabla), f"| días-estación con episodio mañana: {100 * tabla['episodio_manana'].mean():.1f}%")
tabla[["fecha", "estacion", "pm25", "pm25_ayer", "media_7d", "mes", "pm25_manana", "episodio_manana"]].head(3)

## 2. Features sin fuga de datos

El error más caro de la ciencia de datos: entrenar con una variable que **es** la respuesta o que **viene del futuro**. `pm25_manana` es exactamente lo que queremos predecir: si la dejas, el modelo saca F1 ≈ 0,99 en la prueba — demasiado bueno para ser verdad — y en producción no existirá. Elige solo lo que se conoce **hoy**.

### ✍️ Ejercicio 1 — Define FEATURES y el objetivo — sin fuga

In [ ]:
# TODO: lista de columnas que se conocen HOY (de: estacion, pm25, pm25_ayer, media_7d, mes, pm25_manana)
FEATURES = ...
y = tabla["episodio_manana"]
X = tabla[FEATURES] if isinstance(FEATURES, list) else None
print("features:", FEATURES)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert isinstance(FEATURES, list) and len(FEATURES) >= 3
    assert "pm25_manana" not in FEATURES and "episodio_manana" not in FEATURES, "¡fuga! hay una columna del futuro"
    assert set(FEATURES) <= set(tabla.columns)
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Usa estacion, pm25, pm25_ayer, media_7d y mes; nunca pm25_manana ni episodio_manana.')
    print("   Detalle:", e)

## 3. Partición temporal

Con datos en el tiempo, la prueba debe estar **después** del entrenamiento: entrena con 2020–2024 y prueba con 2025. Un `train_test_split` al azar mezclaría días vecinos entre entrenamiento y prueba (el PM2.5 de hoy se parece mucho al de ayer) y daría una idea optimista de lo que pasará con datos realmente nuevos.

### ✍️ Ejercicio 2 — Separa entrenamiento (≤ 2024) y prueba (2025)

In [ ]:
# TODO: máscara True para las filas de 2025 en adelante
en_prueba = ...
# TODO: separa X e y con la máscara (entrenamiento = ~en_prueba)
Xtr, ytr = ..., ...
Xte, yte = ..., ...

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    _m = tabla["fecha"] >= "2025-01-01"
    assert len(Xtr) == int((~_m).sum()) and len(Xte) == int(_m.sum())
    assert tabla.loc[Xtr.index, "fecha"].max() < tabla.loc[Xte.index, "fecha"].min(), "la prueba debe ser posterior"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'en_prueba = tabla["fecha"] >= "2025-01-01"; Xtr, ytr = X[~en_prueba], y[~en_prueba]; Xte, yte = X[en_prueba], y[en_prueba].')
    print("   Detalle:", e)

## 4. Entrena el modelo

Un **pipeline** con codificación one-hot de las categóricas (estación y mes) y escalado de las numéricas, más una **regresión logística**. El pipeline aprende el preprocesamiento solo con el entrenamiento: sin fuga por la puerta de atrás.

### ✍️ Ejercicio 3 — Entrena el pipeline

In [ ]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["estacion", "mes"]),
    ("num", StandardScaler(), ["pm25", "pm25_ayer", "media_7d"]),
])
# TODO: Pipeline(pre + LogisticRegression(max_iter=1000)) entrenado con Xtr, ytr
modelo = ...
print("modelo entrenado:", hasattr(modelo, "classes_"))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert hasattr(modelo, "predict") and hasattr(modelo, "classes_"), "falta el .fit(Xtr, ytr)"
    assert list(modelo.classes_) == [0, 1]
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))]).fit(Xtr, ytr).")
    print("   Detalle:", e)

## 5. Evalúa honestamente: contra la línea base

Los episodios son pocos (≈3% de los días-estación de 2025): la *accuracy* engaña, usa **F1** y ***recall***. Y un modelo no se evalúa solo: se compara con la **regla simple** que ya usaría cualquiera — «si hoy superó 50, mañana también» (persistencia).

### ✍️ Ejercicio 4 — F1 del modelo vs F1 de la persistencia

In [ ]:
pred = modelo.predict(Xte)
# TODO: F1 y recall del modelo en la prueba
f1_modelo = ...
recall_modelo = ...
# TODO: línea base de persistencia: 1 si el pm25 de HOY (Xte["pm25"]) supera 50; y su F1
base = ...
f1_base = ...
print("F1 modelo:", f1_modelo, "| F1 persistencia:", f1_base)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    _p = modelo.predict(Xte)
    assert abs(f1_modelo - f1_score(yte, _p)) < 1e-9 and abs(recall_modelo - recall_score(yte, _p)) < 1e-9
    assert abs(f1_base - f1_score(yte, (Xte["pm25"] > 50).astype(int))) < 1e-9
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'f1_score(yte, pred), recall_score(yte, pred); base = (Xte["pm25"] > 50).astype(int); f1_score(yte, base).')
    print("   Detalle:", e)

## 6. Ficha del modelo (*model card*)

Completa esta ficha con **tus** números (los del ejemplo están entre paréntesis). Acompaña al
modelo siempre que lo entregues:

> **Nombre / versión:** _(p. ej. alerta-pm25-manana v1)_
> **Decisión que apoya y usuarios:** _(quién lo usa, para qué; con revisión humana)_
> **Datos:** _(fuente, período, unidad de análisis; ejemplo: SINCA, 14 estaciones, 2020–2025, una fila = estación-día)_
> **Partición:** _(ejemplo: entrena 2020–2024, prueba 2025 — temporal, sin barajar)_
> **Desempeño:** F1 = ___ ; recall = ___ ; **línea base** (persistencia) F1 = ___
> **¿Le gana a la línea base?** _(dilo con el número; en el ejemplo, por muy poco)_
> **Limitaciones y sesgos:** _(¿en qué estaciones/grupos rinde peor? ¿qué datos no usa?)_
> **No usar para:** _(p. ej. decretar episodios de forma automática)_
> **Responsable y monitoreo:** _(quién responde; qué se vigila y cada cuánto — ver R2-09)_

## 7. Brief para quien decide (una plana)

> **Pregunta:** _(la del paso 1)_
>
> **Resultado (1–2 frases, con el número clave):** _(p. ej. «el modelo detecta X de cada 10 episodios con un día de aviso, casi igual que la regla simple»)_
>
> **Recomendación:** _(¿se usa?, ¿cómo?, ¿qué falta para que aporte más? p. ej. agregar meteorología)_
>
> **Límites y ética:** _(qué NO dice; a quién podría afectar un error; revisión humana)_

## Rúbrica de evaluación

| Criterio | Qué se evalúa |
|---|---|
| **Framing** | La decisión pública, el usuario y el objetivo del modelo están bien planteados. |
| **Calidad del dato** | Limpieza, unidad de análisis clara y *features* razonables. |
| **Evaluación honesta** | Sin **fuga de datos**; partición adecuada (temporal si hay fechas); métrica acorde al desbalance; **comparación con una línea base**. |
| **Ficha de modelo** | Documenta datos, desempeño, límites, usos no recomendados y responsable. |
| **Reproducibilidad** | El notebook corre de principio a fin sin pasos manuales. |
| **Comunicación** | El brief es claro, honesto y accionable para alguien no técnico. |
| **Pertinencia** | (Bonus) El dataset es del propio organismo o responde una pregunta real de gestión. |

> **Entregable:** notebook reproducible + modelo entrenado + **ficha de modelo** + **brief** de una plana.